<a href="https://colab.research.google.com/github/DivyAgrawal11/Macro-Regime-Asset-Allocation/blob/main/Macro_Regime_Model_India.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries silently
!pip install fredapi yfinance plotly pandas numpy > /dev/null

import pandas as pd
import numpy as np
import yfinance as yf
from fredapi import Fred
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully. Ready for data ingestion.")

Libraries imported successfully. Ready for data ingestion.


In [3]:
# --- INPUT YOUR FRED API KEY HERE ---
FRED_API_KEY = 'YOUR_API_KEY_HERE'
fred = Fred(api_key=FRED_API_KEY)

def fetch_macro_data():
    print("1. Fetching Macro Data from FRED...")

    # Using the most current, verified FRED tickers for India
    print(" -> Fetching CPI...")
    cpi = fred.get_series('INDCPALTT01IXNBM', observation_start='2010-01-01')

    print(" -> Fetching 10Y Yield...")
    yield_10y = fred.get_series('INDIRLTLT01STM', observation_start='2010-01-01')

    print(" -> Fetching Short-Term Rate...")
    yield_3m = fred.get_series('IRSTPI01INM156N', observation_start='2010-01-01')

    print(" -> Fetching Industrial Production...")
    ind_prod = fred.get_series('INDPROMANMISMEI', observation_start='2010-01-01')

    macro_df = pd.DataFrame({
        'CPI': cpi,
        'Yield_10Y': yield_10y,
        'Yield_3M': yield_3m,
        'Industrial_Production': ind_prod
    })
    print(f"-> Successfully pulled {len(macro_df)} months of macro data.")
    return macro_df.resample('M').last().ffill()

def fetch_market_data():
    print("\n2. Fetching NIFTY 50 Data from YFinance...")
    nifty = yf.Ticker('^NSEI').history(start='2010-01-01', interval='1mo')

    if nifty.empty:
        raise ValueError("YFinance returned no data.")

    market_df = pd.DataFrame({'Nifty_50': nifty['Close']})
    market_df.index = market_df.index.tz_localize(None)
    print(f"-> Successfully pulled {len(market_df)} months of market data.")
    return market_df.resample('M').last().ffill()

# Execute functions
macro_data = fetch_macro_data()
market_data = fetch_market_data()

# Merge macro and market data together
df = pd.merge(macro_data, market_data, left_index=True, right_index=True, how='inner')
print("\n--- DATA INGESTION COMPLETE ---")
display(df.head())

1. Fetching Macro Data from FRED...
 -> Fetching CPI...
 -> Fetching 10Y Yield...
 -> Fetching Short-Term Rate...
 -> Fetching Industrial Production...
-> Successfully pulled 197 months of macro data.

2. Fetching NIFTY 50 Data from YFinance...
-> Successfully pulled 198 months of market data.

--- DATA INGESTION COMPLETE ---


,CPI,Yield_10Y,Yield_3M,Industrial_Production,Nifty_50
2010-01-31,59.7184,NaN,11.5,86.39656,4882.049805
2010-02-28,59.0240,NaN,11.5,87.25145,4922.299805
2010-03-31,59.0240,NaN,11.5,87.31899,5249.100098
2010-04-30,59.0240,NaN,11.5,88.50832,5278.000000
2010-05-31,59.7184,NaN,11.5,85.89028,5086.299805


In [4]:
# 1. Feature Engineering
# Calculate Year-over-Year (YoY) percentage changes
df['CPI_YoY'] = df['CPI'].pct_change(12) * 100
df['IndProd_YoY'] = df['Industrial_Production'].pct_change(12) * 100
df['Nifty_YoY'] = df['Nifty_50'].pct_change(12) * 100

# The Yield Spread: 10-Year minus 3-Month. A key indicator of credit conditions.
df['Yield_Spread'] = df['Yield_10Y'] - df['Yield_3M']

# Smooth the data with a 6-month moving average to filter out market noise
df['Growth_Trend'] = df['IndProd_YoY'].rolling(window=6).mean()
df['Inflation_Trend'] = df['CPI_YoY'].rolling(window=6).mean()

# Drop rows with NaN values created by the rolling windows
df.dropna(inplace=True)

# 2. Regime Classification Algorithm
def classify_regime(row):
    """
    Classifies the macroeconomic regime based on Growth and Inflation momentum.
    - Expansion: Growth accelerating, Inflation stable/falling
    - Slowdown: Growth decelerating, Inflation rising
    - Contraction: Growth decelerating, Inflation falling
    - Recovery: Growth accelerating, Inflation falling
    """
    # Compare current value to its recent 6-month trend
    growth_accelerating = row['IndProd_YoY'] > row['Growth_Trend']
    inflation_rising = row['CPI_YoY'] > row['Inflation_Trend']

    if growth_accelerating and not inflation_rising:
        return 'Recovery'
    elif growth_accelerating and inflation_rising:
        return 'Expansion'
    elif not growth_accelerating and inflation_rising:
        return 'Slowdown'
    else:
        return 'Contraction'

# Apply the model to our dataset
df['Macro_Regime'] = df.apply(classify_regime, axis=1)

print("Feature engineering and Regime Classification complete.")
display(df[['CPI_YoY', 'IndProd_YoY', 'Yield_Spread', 'Macro_Regime']].tail())

Feature engineering and Regime Classification complete.


,CPI_YoY,IndProd_YoY,Yield_Spread,Macro_Regime
2026-01-31,0.0,0.0,-2.838,Contraction
2026-02-28,0.0,0.0,-2.800,Contraction
2026-03-31,0.0,0.0,-2.730,Contraction
2026-04-30,0.0,0.0,-2.520,Contraction
2026-05-31,0.0,0.0,-2.550,Contraction


In [5]:
# Create a professional, multi-panel interactive chart
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    vertical_spacing=0.1,
                    subplot_titles=('NIFTY 50 with Macro Regimes', 'India Yield Spread (10Y - 3M)'),
                    row_heights=[0.7, 0.3])

# Define colors for our business cycle phases
regime_colors = {
    'Recovery': 'rgba(0, 255, 0, 0.2)',    # Green
    'Expansion': 'rgba(0, 0, 255, 0.2)',   # Blue
    'Slowdown': 'rgba(255, 165, 0, 0.2)',  # Orange
    'Contraction': 'rgba(255, 0, 0, 0.2)'  # Red
}

# Plot NIFTY 50
fig.add_trace(go.Scatter(x=df.index, y=df['Nifty_50'], name='NIFTY 50', line=dict(color='black', width=2)), row=1, col=1)

# Overlay Regimes as background color blocks
for regime, color in regime_colors.items():
    regime_data = df[df['Macro_Regime'] == regime]
    for date in regime_data.index:
        fig.add_vrect(
            x0=date, x1=date + pd.Timedelta(days=30),
            fillcolor=color, opacity=0.5, layer="below", line_width=0,
            row=1, col=1
        )

# Create a dummy trace for the legend so viewers know what the colors mean
for regime, color in regime_colors.items():
    fig.add_trace(go.Bar(x=[None], y=[None], marker=dict(color=color), name=regime), row=1, col=1)

# Plot Yield Spread on the second row
fig.add_trace(go.Scatter(x=df.index, y=df['Yield_Spread'], name='Yield Spread (10Y-3M)', line=dict(color='purple', width=2)), row=2, col=1)
# Add a zero line for the yield spread (Inversion indicator)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=2, col=1)

# Update layout for a clean, professional look
fig.update_layout(
    title='Indian Market Macroeconomic Dashboard',
    height=800,
    plot_bgcolor='white',
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

# Display the interactive dashboard inside Colab
fig.show()

# Print Strategic Allocation Table
print("\n--- Strategic Asset Allocation Guide (Indian Market) ---")
allocation = pd.DataFrame({
    'Regime': ['Recovery', 'Expansion', 'Slowdown', 'Contraction'],
    'Equity (NIFTY 50)': ['Overweight (High Beta)', 'Overweight (Quality)', 'Underweight', 'Underweight (Defensive)'],
    'Fixed Income (Bonds)': ['Underweight', 'Neutral', 'Overweight (Short Duration)', 'Overweight (Long Duration)'],
    'Cash/Gold': ['Underweight', 'Neutral', 'Overweight', 'Overweight']
})
display(allocation)


--- Strategic Asset Allocation Guide (Indian Market) ---


,Regime,Equity (NIFTY 50),Fixed Income (Bonds),Cash/Gold
0,Recovery,Overweight (High Beta),Underweight,Underweight
1,Expansion,Overweight (Quality),Neutral,Neutral
2,Slowdown,Underweight,Overweight (Short Duration),Overweight
3,Contraction,Underweight (Defensive),Overweight (Long Duration),Overweight
